In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

/home/fede/.venvs/labo3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_pickle("df_fe_epic_light_grouped.pickle")


In [3]:
    # 📄 Leer lista de productos a predecir
with open("product_id_apredecir201912.txt", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

In [4]:
df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M')) 
df = df.rename(columns={'fecha': 'timestamp'})
df.drop(columns=["target"], inplace=True, errors='ignore')

In [5]:
# Filtrar hasta dic 2019 y productos requeridos
df = df[
    (df['product_id'].isin(product_ids))
]
df

,product_id,timestamp,cust_request_qty,cust_request_qty_agg_std,cust_request_qty_agg_mean,cust_request_qty_agg_max,cust_request_tn,tn,tn_agg_std,tn_agg_mean,...,cust_request_qty_sku_size_vendidas_div,cust_request_qty_product_id_vendidas,cust_request_qty_product_id_vendidas_div,cust_request_qty_customer_id_vendidas,cust_request_qty_customer_id_vendidas_div,tn_customer_vendidas,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight
0,20001,2017-01-31,479,3.718421,1.106236,49,937.727173,934.772217,13.506243,2.158827,...,0.089801,479,1.0,187176,0.002559,34057.316406,34057.316406,1.0,934.772217,2.744703e-02
36,20002,2017-01-31,391,3.148312,0.903002,34,555.186523,550.157043,6.959879,1.270571,...,0.073303,391,1.0,187176,0.002089,34057.316406,34057.316406,1.0,550.157043,1.615386e-02
72,20003,2017-01-31,438,2.854067,1.011547,28,1067.815430,1063.458374,11.015188,2.456024,...,0.321822,438,1.0,187176,0.002340,34057.316406,34057.316406,1.0,1063.458374,3.122555e-02
108,20004,2017-01-31,339,2.361729,0.782910,37,569.373962,555.916138,8.193350,1.283871,...,0.105084,339,1.0,187176,0.001811,34057.316406,34057.316406,1.0,555.916138,1.632296e-02
144,20005,2017-01-31,249,1.754499,0.575058,28,494.600830,494.270111,7.719429,1.141501,...,0.290210,249,1.0,187176,0.001330,34057.316406,34057.316406,1.0,494.270111,1.451289e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31344,21263,2019-12-31,5,0.108050,0.008375,2,0.012700,0.012700,0.000276,0.000021,...,0.002776,5,1.0,123878,0.000040,26217.068359,26217.068359,1.0,0.012700,4.844173e-07
31384,21265,2019-12-31,5,0.091209,0.008375,1,0.050070,0.050070,0.001378,0.000084,...,0.454545,5,1.0,123878,0.000040,26217.068359,26217.068359,1.0,0.050070,1.909825e-06
31394,21266,2019-12-31,6,0.115419,0.010050,2,0.051210,0.051210,0.001381,0.000086,...,0.545455,6,1.0,123878,0.000048,26217.068359,26217.068359,1.0,0.051210,1.953308e-06
31404,21267,2019-12-31,4,0.081648,0.006700,1,0.015690,0.015690,0.000447,0.000026,...,0.002221,4,1.0,123878,0.000032,26217.068359,26217.068359,1.0,0.015690,5.984650e-07


In [6]:
# transformo cat1, cat2, cat3, brand y sku_size a categoricos
df['cat1'] = df['cat1'].astype('category')
df['cat2'] = df['cat2'].astype('category')
df['cat3'] = df['cat3'].astype('category')
df['brand'] = df['brand'].astype('category')
df['sku_size'] = df['sku_size'].astype('category')


In [7]:
static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
static_features_df

,product_id,cat1,cat2,cat3,brand,sku_size
0,20001,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
1,20002,HC,ROPA LAVADO,Liquido,LIMPIEX,3000.0
2,20003,FOODS,ADEREZOS,Mayonesa,NATURA,475.0
3,20004,FOODS,ADEREZOS,Mayonesa,NATURA,240.0
4,20005,FOODS,ADEREZOS,Mayonesa,NATURA,120.0
...,...,...,...,...,...,...
775,21263,PC,CABELLO,SHAMPOO,VICHY,250.0
776,21265,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32.0
777,21266,PC,PIEL1,CUIDADO ESPECIAL,LANCOME,32.0
778,21267,PC,PIEL1,Cara,NIVEA,250.0


In [8]:
# ⏰ 4. Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column='product_id',
    timestamp_column='timestamp',
    static_features_df=static_features_df,

)
ts_data = ts_data.fill_missing_values()

Trying to fill missing values in an unsorted dataframe. It is highly recommended to call `ts_df.sort_index()` before calling `ts_df.fill_missing_values()`


In [ ]:
# ⚙️ 5. Definir y entrenar predictor
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS',  # Frecuencia mensual (Month Start)
    #known_covariates_names=["timestamp_int", "timestamp.year", "timestamp.month", "timestamp.day", "timestamp.dayofweek"]
)

predictor.fit(ts_data, num_val_windows=2, time_limit=60*60)

Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250704_030628'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.99 GB / 15.32 GB (39.1%)
Disk Space Avail:   59.50 GB / 575.67 GB (10.3%)
Setting presets to: bolt_base

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': {'Chronos': {'model_path': 'bolt_base'}},
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': True,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'ME'

In [10]:
# 🔮 6. Generar predicción
forecast = predictor.predict(ts_data)

data with frequency 'ME' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: Chronos[bolt_base]


In [11]:
# Extraer predicción media y filtrar febrero 2020
forecast_mean = forecast['mean'].reset_index()
print(forecast_mean.columns)

Index(['item_id', 'timestamp', 'mean'], dtype='object')


In [12]:
forecast_mean

,item_id,timestamp,mean
0,20001,2020-01-01,1455.144287
1,20001,2020-02-01,1447.588501
2,20002,2020-01-01,1122.668945
3,20002,2020-02-01,1109.127197
4,20003,2020-01-01,800.012146
...,...,...,...
1555,20995,2020-02-01,1.130481
1556,21087,2020-01-01,0.998036
1557,21087,2020-02-01,0.968546
1558,21214,2020-01-01,0.244202


In [13]:
# Tomar solo item_id y la predicción 'mean'
resultado = forecast['mean'].reset_index()[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

# Filtrar solo febrero 2020
resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']

# Renombrar columnas
resultado = resultado[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

In [14]:
# 💾 7. Guardar archivo
resultado.to_csv("predicciones_febrero2020_fecha_v02_01-more-features.csv", index=False)
resultado.head()

,product_id,tn
1,20001,1447.588501
3,20002,1109.127197
5,20003,749.192932
7,20004,526.168152
9,20005,507.228241
